In [33]:
from torch.utils.data import Dataset, DataLoader

In [18]:
import evaluate

In [19]:
import pandas as pd

In [60]:
from peft import PeftModel, PeftConfig
from PIL import Image
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from peft import LoraConfig, get_peft_model, TaskType
from transformers.models.blip import modeling_blip_text

In [21]:
from transformers import BlipProcessor, BlipForConditionalGeneration

In [22]:
from pycocoevalcap.cider.cider import Cider

In [23]:
import evaluate

In [24]:
df = pd.read_csv('data/train_data.csv')

In [57]:
def calculate_cider(generated_captions, ground_truth_captions):
    """
    generated_captions: List of strings
    ground_truth_captions: List of strings
    """
    
    # 1. Format data into dictionaries expected by the library
    # Structure: {image_id: [caption_string]}
    gts = {}
    res = {}
    
    for i, (gen, true) in enumerate(zip(generated_captions, ground_truth_captions)):
        img_id = str(i) # Create a dummy ID for the image
        
        # Ground Truth (Reference)
        # Note: In standard datasets, one image might have 5 captions.
        # In medical data, it usually has 1. We wrap it in a list [].
        gts[img_id] = [true] 
        
        # Generated (Hypothesis)
        res[img_id] = [gen]

    # 2. Initialize Scorer
    scorer = Cider()
    
    # 3. Compute Score
    # score: The average CIDEr score for the whole dataset
    # scores: A list of CIDEr scores for each individual image
    score, scores = scorer.compute_score(gts, res)
    
    return score, scores

# --- Usage Example ---

generated = [
    "no acute cardiopulmonary abnormality",
    "mild cardiomegaly is present"
]

ground_truth = [
    "the lungs are clear no acute disease",
    "heart size is mildly enlarged"
]

# average_score, individual_scores = calculate_cider(df['generated_captions'], df['findings'])

# print(f"Overall CIDEr Score: {average_score:.4f}")

In [30]:
# Train 

In [32]:
df.head()

,uid,filename,findings,impression,Problems
0,2,2_IM-0652-1001.dcm.png,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.,Cardiomegaly;Pulmonary Artery
1,4,4_IM-2050-1001.dcm.png,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...,"Pulmonary Disease, Chronic Obstructive;Bullous..."
2,5,5_IM-2117-1003002.dcm.png,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.,Osteophyte;Thickening;Lung
3,6,6_IM-2192-1001.dcm.png,Heart size and mediastinal contour are within ...,No acute cardiopulmonary findings.,normal
4,7,7_IM-2263-1001.dcm.png,The cardiac contours are normal. XXXX basilar ...,Basilar atelectasis. No confluent lobar consol...,Pulmonary Atelectasis;Spondylosis;Arthritis


In [34]:
class ChestIUDataset(Dataset):
    def __init__(self, image_dir, report_df, processor):
        self.image_dir = image_dir
        self.report_df = report_df
        self.processor = processor
        self.samples = self._load_samples()

    def _load_samples(self):
        samples = []
        for i in range(len(self.report_df)):
            samples.append({"image": self.image_dir + "/" + str(self.report_df['filename'].iloc[i]), "caption": self.report_df['findings'].iloc[i]})
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["image"]).convert("RGB")
        caption = item["caption"]

        # Preprocess using BLIP's processor
        encoding = self.processor(
            images=image, 
            text=caption, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        
        # Remove batch dimension added by processor
        return {k: v.squeeze(0) for k, v in encoding.items()}

In [ ]:
# Load Base Model
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
# 1. Enable gradients for input embeddings (Crucial step)
model.enable_input_require_grads()

# 2. Configure LoRA (Without task_type!)
config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["query", "value"],
    lora_dropout=0.05, 
    bias="none"
)

# 3. Apply LoRA
model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 225,151,292 || trainable%: 0.5239


In [ ]:
# Define the fixed forward function
def fixed_forward(self, input_ids=None, position_ids=None, inputs_embeds=None, past_key_values_length=0):
    if inputs_embeds is None:
        inputs_embeds = self.word_embeddings(input_ids)
    
    embeddings = inputs_embeds

    if self.position_embedding_type == "absolute":
        if position_ids is None:
            # Recreating the logic from the original library to get position_ids
            if input_ids is not None:
                seq_length = input_ids.shape[1]
                position_ids = torch.arange(past_key_values_length, seq_length + past_key_values_length, dtype=torch.long, device=embeddings.device)
                position_ids = position_ids.unsqueeze(0).expand(input_ids.shape[:2])
            else:
                position_ids = self.create_position_ids_from_inputs_embeds(inputs_embeds)

        position_embeddings = self.position_embeddings(position_ids)
        
        # --- THE FIX IS HERE ---
        # Changed from `embeddings += position_embeddings` to out-of-place addition
        embeddings = embeddings + position_embeddings 
        # -----------------------

    embeddings = self.LayerNorm(embeddings)
    embeddings = self.dropout(embeddings)
    return embeddings

# Apply the patch
modeling_blip_text.BlipTextEmbeddings.forward = fixed_forward

In [ ]:
# Hyperparameters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4 # Small batch size because images are large (384x384)
EPOCHS = 5
LR = 5e-5

# Load Data
dataset = ChestIUDataset("data/images/images_normalized", df, processor)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Optimizer
optimizer = AdamW(model.parameters(), lr=LR)

# Move model to GPU
model.to(DEVICE)
model.train()

print("Starting Training...")

for epoch in range(EPOCHS):
    epoch_loss = 0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
    
    for batch in progress_bar:
        # Move batch to device
        input_ids = batch["input_ids"].to(DEVICE)
        pixel_values = batch["pixel_values"].to(DEVICE)
        
        # BLIP Forward Pass
        # Note: We pass input_ids as 'labels' for the model to learn to generate them
        outputs = model(
            input_ids=input_ids, 
            pixel_values=pixel_values, 
            labels=input_ids
        )
        
        loss = outputs.loss
        loss.backward()
        
        optimizer.step()
        optimizer.zero_grad()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
        
    print(f"Epoch {epoch+1} Average Loss: {epoch_loss/len(dataloader)}")

# Save the adapter weights
model.save_pretrained("blip-chest-xray-lora")

Starting Training...


Epoch 1: 100%|██████████| 580/580 [04:50<00:00,  1.99it/s, loss=7.69]


Epoch 1 Average Loss: 9.049614649805529


Epoch 2: 100%|██████████| 580/580 [04:51<00:00,  1.99it/s, loss=7.37]


Epoch 2 Average Loss: 7.593623549362709


...

Epoch 3 Average Loss: 7.479863245733853


Epoch 4: 100%|██████████| 580/580 [04:50<00:00,  2.00it/s, loss=7.36]


Epoch 4 Average Loss: 7.429384527535274


Epoch 5: 100%|██████████| 580/580 [04:51<00:00,  1.99it/s, loss=7.37]


Epoch 5 Average Loss: 7.399948425950675


In [39]:
df_test = pd.read_csv('data/test_data.csv')

In [41]:
base_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = PeftModel.from_pretrained(base_model, "blip-chest-xray-lora")
model.to("cuda")
model.eval()
ls = []
for i in tqdm(range(len(df_test))):
    image = Image.open(f"data/images/images_normalized/{df_test.filename.iloc[i]}").convert('RGB')
    inputs = processor(images=image, return_tensors="pt").to("cuda")
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=50)
        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        ls.append(caption)
df_test['generated_captions'] = ls

100%|██████████| 491/491 [04:04<00:00,  2.00it/s]


In [54]:
df_test['generated_captions'].head(5)

0    the heart is normal in size and contours are w...
1    the heart is normal in size and contours are w...
2    the heart is normal in size and contours are w...
3    the heart is normal in size and contours are w...
4    the heart size is normal in size and contours ...
Name: generated_captions, dtype: object

In [55]:
df_test['generated_captions'].iloc[0]

'the heart is normal in size and contours are within normal limits for size and contours, pleural effusion, or pneumothorax, and mediastinum of the thoracic spine, and mediasti'

In [56]:
df_test['findings'].iloc[0]

'Heart size within normal limits, stable mediastinal and hilar contours. Mild hyperinflation appears similar to prior. No focal alveolar consolidation, no definite pleural effusion seen. Scattered chronic appearing irregular interstitial markings, no typical findings of pulmonary edema.'

In [51]:
bleu_metric = evaluate.load("bleu")

In [52]:
results = bleu_metric.compute(predictions=df_test['generated_captions'], references=df_test['findings'], max_order=4)
print(f"BLEU-4: {results}")
average_score, individual_scores = calculate_cider(df_test['generated_captions'], df_test['findings'])
print(f"CIDEr: {average_score}")

BLEU-4: {'bleu': 0.0716446667477945, 'precisions': [0.3035789221269708, 0.10549722050648548, 0.04567169883432066, 0.02235665439242504], 'brevity_penalty': 0.9474192704484686, 'length_ratio': 0.948754407917188, 'translation_length': 16681, 'reference_length': 17582}
CIDEr: 0.10771264148269286
